In [ ]:
#!/usr/bin/env python3

import json


def ffm_to_geojson(input_file, output_file):
    features = []

    with open(input_file) as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):

        line = lines[i]

        if line.startswith("#Boundary of Fault_segment"):

            # Try to extract segment number
            try:
                segment = int(line.split()[3].rstrip("."))
            except Exception:
                segment = len(features) + 1

            # Advance until we find the coordinate header
            i += 1
            while i < len(lines) and "Lon." not in lines[i]:
                i += 1

            if i >= len(lines):
                break

            # Skip header
            i += 1

            coords = []

            # Read the five polygon vertices
            for _ in range(5):
                parts = lines[i].split()
                lon = float(parts[0])
                lat = float(parts[1])
                coords.append([lon, lat])
                i += 1

            feature = {
                "type": "Feature",
                "properties": {
                    "segment": segment
                },
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [coords]
                }
            }

            features.append(feature)

            continue

        i += 1

    geojson = {
        "type": "FeatureCollection",
        "features": features
    }

    with open(output_file, "w") as f:
        json.dump(geojson, f, indent=2)

    print(f"Wrote {len(features)} polygons to {output_file}")


if __name__ == "__main__":
    ffmdir='/Users/hyin/usgs_mendenhall/events/2026-04-14_nevada/wisp_inversion/ffm_results/NP2.2/'
    ffm_to_geojson(f"{ffmdir}/Solution.txt", f"{ffmdir}/fault_segments.geojson")

Wrote 2 polygons to /Users/hyin/usgs_mendenhall/events/2026-06-28_venezuela/ffm/20260624_Venezuela_V3/TwoEvent.42//fault_segments.geojson
